In [49]:
import pandas as pd
import numpy as np 
import networkx as nx

import json
import re

In [50]:
df_int = pd.read_csv("../../dataset/final_dataset/contacted_anon.csv")

# Only keep vacancies with at least 15 interactions
relevant_vacancies = df_int["cvid"].value_counts()[df_int["cvid"].value_counts() >= 15].index

# Filter to only relevant vacancies
df_int = df_int[df_int["cvid"].isin(relevant_vacancies.values)]

# Load triples
df_vacancy = pd.read_excel(f"../outputs/final_outputs/triples.xlsx").drop("Unnamed: 0", axis=1)
df_cv = pd.read_excel("../outputs/final_outputs/cv.xlsx").drop("Unnamed: 0", axis=1)

In [51]:
bridges = pd.read_excel("../outputs/final_outputs/bridges.xlsx")

In [52]:
def clean_entity(entity):
    """Normalizes a string to lowercase, replaces spaces/hyphens with underscores, and strips symbols."""
    # Return as-is if the entity is a float/int (like 28.0)
    if not isinstance(entity, str):
        return entity
        
    # Lowercase the string
    e = entity.lower()
    
    # Remove specific symbols like '&' entirely before replacing other chars
    e = e.replace('&', '')
    
    # Replace anything that isn't a letter, number, or underscore with a space
    e = re.sub(r'[^a-z0-9_]', ' ', e)
    
    # Replace multiple spaces with a single underscore, and strip leading/trailing spaces
    e = re.sub(r'\s+', '_', e.strip())
    
    # Remove leading or trailing underscores just in case
    e = e.strip('_')
    
    return e

def clean_triples(triples):
    """Applies the cleaning function to the subject and object of each triple."""
    cleaned = []
    for sub, pred, obj in triples:
        cleaned.append((clean_entity(sub), pred, clean_entity(obj)))
    return cleaned

In [53]:
is_connected = []

for row in df_int.iterrows():
    job, cvid = row[1][0], row[1][2]

    vacancy_triples = clean_triples(eval(df_vacancy[df_vacancy["id"] == job]["triples_qwen_structured"].values[0]))
    cv_triples = clean_triples(eval(df_cv[df_cv["id"] == cvid]["CV_triples"].values[0]))
    try:
        bridge_triples = clean_triples(eval(bridges[(bridges["humanjobid"] == job) & 
            (bridges["cvid"] == cvid)]["bridges_qwen_structured"].values[0]))
    except TypeError:
        continue

    G = nx.Graph()

    G.add_edges_from([(s, o) for s, _, o in vacancy_triples])
    G.add_edges_from([(s, o) for s, _, o in cv_triples])
    G.add_edges_from([(s, o) for s, _, o in bridge_triples])

    print(np.mean(is_connected), end="\r")

C:\Users\roans\AppData\Local\Temp\ipykernel_11016\1549743382.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  job, cvid = row[1][0], row[1][2]


0.66666666666666664

KeyboardInterrupt: 

In [41]:
bridges[(bridges["humanjobid"] == 1527507) & 
        (bridges["cvid"] == "de78ead1ba7f4b5aaa37428f246b622e")]["bridges_qwen_structured"].values[0]

nan